# Policy Scenario Analysis: Great Miami Hurricane

This notebook visualizes Monte Carlo results for policy interventions under the Great Miami hurricane scenario.

**Analysis Structure:**
- **Table 1:** Baseline hazard stress (current climate)
- **Figure 1:** Loss decomposition by peril/institution
- **Table 2:** Climate change amplification (SSP2-4.5, 2070-2100)
- **Table 3:** Policy scenario comparison (current climate)
- **Figure 3:** Policy scenario dashboard
- **Figure 4:** Climate-policy interactions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

## 0. Load Baseline Data from Excel

Load the baseline scenario results from the Excel report to create Figure 1.

In [ ]:
# Load baseline data from Excel report
BASELINE_FILE = Path("../results/mc_runs/scenario_report_with_uncertainty_20260211_172703.xlsx")
#BASELINE_FILE = Path("../results/mc_runs/mc_20251112_110229/scenario_report_with_uncertainty.xlsx")

# Read the summary sheet
xl = pd.ExcelFile(BASELINE_FILE)
print("Available sheets:")
for sheet in xl.sheet_names:
    print(f"  - {sheet}")
print()

# Load the mean values (baseline scenario)
df_baseline = pd.read_excel(BASELINE_FILE, sheet_name='Mean Values')
print("Baseline data loaded:")
print(df_baseline.head(20))
print(f"\nShape: {df_baseline.shape}")
print(f"\nColumns: {df_baseline.columns.tolist()}")

## Figure 2: Multi-Scenario Comparison with Uncertainty

In [ ]:
# Extract data for all scenarios
scenarios = ['Great Miami', 'Andrew', 'Double Great Miami', 'Andrew then Great Miami', 
             'Great Miami then Andrew', 'Lake Okeechobee', 'Irma', 'Double Irma']

# Prepare data dictionary
scenario_data = {}
for scenario in scenarios:
    if scenario in df_baseline.columns:
        metrics = dict(zip(df_baseline['Metric'], df_baseline[scenario]))
        
        # Calculate components (in billions)
        wind_gross = metrics['Wind loss gross (USD)'] / 1e9
        flood_gross = metrics['Flood loss gross (USD)'] / 1e9
        citizens_gross = metrics['Citizens loss gross (USD)'] / 1e9
        wind_under = metrics['Un/underinsured wind (USD)'] / 1e9
        flood_under = metrics['Un/underinsured flood (USD)'] / 1e9
        
        # Insured wind (private) = Wind gross - Citizens - Un/underinsured wind
        insured_wind = wind_gross - citizens_gross - wind_under
        # Insured flood = Flood gross - Un/underinsured flood
        insured_flood = flood_gross - flood_under
        
        scenario_data[scenario] = {
            'insured_wind': insured_wind,
            'citizens': citizens_gross,
            'insured_flood': insured_flood,
            'wind_under': wind_under,
            'flood_under': flood_under
        }

# Display the extracted data
print("Scenario data extracted (in billions USD):")
print(f"\n{'Scenario':<25} {'Insured Wind':<12} {'Citizens':<12} {'Insured Flood':<12} {'Wind U/U':<12} {'Flood U/U':<12}")
print("-" * 85)
for scenario, data in scenario_data.items():
    print(f"{scenario:<25} ${data['insured_wind']:>10.1f}B  ${data['citizens']:>10.1f}B  ${data['insured_flood']:>10.1f}B  ${data['wind_under']:>10.1f}B  ${data['flood_under']:>10.1f}B")

In [ ]:
# Create 2-panel figure: Loss decomposition (a) and Institutional stress (b)
fig = plt.figure(figsize=(10, 3))
gs = fig.add_gridspec(1, 2, wspace=0.2)
ax1 = fig.add_subplot(gs[0, 0])  # Panel a: Loss decomposition
ax2 = fig.add_subplot(gs[0, 1])  # Panel b: Institutional stress

scenario_order = ['Great Miami', 'Andrew', 'Lake Okeechobee', 'Irma', 
                  'Great Miami then Andrew', 'Double Great Miami', 'Double Irma']

# ===== PANEL A: LOSS DECOMPOSITION =====
insured_wind_vals = [scenario_data[s]['insured_wind'] for s in scenario_order]
citizens_vals = [scenario_data[s]['citizens'] for s in scenario_order]
insured_flood_vals = [scenario_data[s]['insured_flood'] for s in scenario_order]
wind_under_vals = [scenario_data[s]['wind_under'] for s in scenario_order]
flood_under_vals = [scenario_data[s]['flood_under'] for s in scenario_order]

x_pos = np.arange(len(scenario_order))
width = 0.3

# Create stacked bars with intuitive colors: red for wind, blue for flood, orange for Citizens, grays for underinsured
p1 = ax1.bar(x_pos, insured_wind_vals, width, label='Insured Wind (Private)', color='#E74C3C')
p2 = ax1.bar(x_pos, citizens_vals, width, bottom=insured_wind_vals, 
            label='Citizens Wind', color='#F39C12')
bottom2 = np.array(insured_wind_vals) + np.array(citizens_vals)
p3 = ax1.bar(x_pos, insured_flood_vals, width, bottom=bottom2,
            label='Insured Flood (NFIP)', color='#3498DB')
bottom3 = bottom2 + np.array(insured_flood_vals)
p4 = ax1.bar(x_pos, wind_under_vals, width, bottom=bottom3,
            label='Un/Underinsured Wind', color='#95A5A6')
bottom4 = bottom3 + np.array(wind_under_vals)
p5 = ax1.bar(x_pos, flood_under_vals, width, bottom=bottom4,
            label='Un/Underinsured Flood', color='#BDC3C7')

# Calculate total losses and percentages
totals = (np.array(insured_wind_vals) + np.array(citizens_vals) + 
          np.array(insured_flood_vals) + np.array(wind_under_vals) + 
          np.array(flood_under_vals))

# Calculate two-way split: insured vs uninsured
insured_total = np.array(insured_wind_vals) + np.array(citizens_vals) + np.array(insured_flood_vals)
uninsured_total = np.array(wind_under_vals) + np.array(flood_under_vals)

insured_pct = (insured_total / totals) * 100
uninsured_pct = (uninsured_total / totals) * 100

# Add horizontal line and percentage labels
for i in range(len(scenario_order)):
    # Line separating insured from uninsured
    y_line = bottom3[i]  # Top of insured section
    ax1.plot([i - width/2, i + width/2], [y_line, y_line], 
            color='black', linewidth=1.5, linestyle='-', alpha=0.7)
    
    # Percentage labels
    # Insured percentage (middle of insured section)
    y_insured_mid = insured_total[i] / 2
    ax1.text(i + width/2 + 0.05, y_insured_mid, f'{insured_pct[i]:.0f}%', 
            ha='left', va='center', fontsize=7, fontweight='bold', color='#2C3E50')
    
    # Uninsured percentage (middle of uninsured section)
    y_uninsured_mid = bottom3[i] + uninsured_total[i] / 2
    ax1.text(i + width/2 + 0.05, y_uninsured_mid, f'{uninsured_pct[i]:.0f}%', 
            ha='left', va='center', fontsize=7, fontweight='bold', color='#7F8C8D')

# Add total labels on top of each bar
for i, total in enumerate(totals):
    ax1.text(i, total + 5, f'${total:.0f}B', ha='center', va='bottom', 
            fontweight='bold', fontsize=9)

# Formatting Panel A
ax1.set_ylabel('Loss (Billion USD)', fontsize=11, fontweight='bold')
#ax1.set_title('Loss Decomposition', fontsize=11, fontweight='bold', pad=10)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(scenario_order, rotation=45, ha='right', fontsize=9)
ax1.set_yticks(np.arange(0, 350, 50))
ax1.tick_params(axis='both', which='major', length=4, width=1)
ax1.legend(loc='upper left', frameon=True, fontsize=8, ncol=1)
ax1.grid(False)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['left'].set_color('k')
ax1.spines['bottom'].set_color('k')
ax1.tick_params(axis='y', which='both', left=True, right=False)
ax1.text(-0.1, 1.05, 'a', transform=ax1.transAxes, fontsize=12, fontweight='bold', va='top')

# ===== PANEL B: INSTITUTIONAL STRESS =====
# Extract institutional stress data for all scenarios
institutional_data = {}
for scenario in scenario_order:
    if scenario in df_baseline.columns:
        metrics = dict(zip(df_baseline['Metric'], df_baseline[scenario]))
        
        # Extract institutional stress metrics (in billions)
        fhcf_shortfall = metrics.get('FHCF shortfall (USD)', 0) / 1e9
        figa_residual = metrics.get('FIGA residual (USD)', 0) / 1e9
        citizens_deficit = metrics.get('Citizens residual (USD)', 0) / 1e9
        nfip_borrowed = metrics.get('NFIP Treasury borrowing (USD)', 0) / 1e9
        
        institutional_data[scenario] = {
            'fhcf_shortfall': fhcf_shortfall,
            'figa_residual': figa_residual,
            'citizens_deficit': citizens_deficit,
            'nfip_borrowed': nfip_borrowed
        }
fhcf_vals = np.array([institutional_data[s]['fhcf_shortfall'] for s in scenario_order])
figa_vals = np.array([institutional_data[s]['figa_residual'] for s in scenario_order])
citizens_vals_inst = np.array([institutional_data[s]['citizens_deficit'] for s in scenario_order])
nfip_vals = np.array([institutional_data[s]['nfip_borrowed'] for s in scenario_order])

# Calculate total public burden
total_public = fhcf_vals + figa_vals + citizens_vals_inst + nfip_vals

# Define colors for institutional stress
color_fhcf = "#DCB7EB"      # Purple for FHCF
color_figa = '#8E44AD'      # Darker purple for FIGA
color_citizens = '#F39C12'  # Orange for Citizens (matching previous)
color_nfip = '#3498DB'      # Blue for NFIP (matching flood)

# Create stacked bars)
bars1 = ax2.bar(x_pos, fhcf_vals, width, label='FHCF Shortfall', color=color_fhcf)
bars2 = ax2.bar(x_pos, figa_vals, width, bottom=fhcf_vals, label='FIGA Residual', color=color_figa)
bars3 = ax2.bar(x_pos, citizens_vals_inst, width, bottom=fhcf_vals + figa_vals, label='Citizens Deficit', color=color_citizens)
bars4 = ax2.bar(x_pos, nfip_vals, width, bottom=fhcf_vals + figa_vals + citizens_vals_inst, label='NFIP Borrowing', color=color_nfip)

# Add total public burden labels on top
for i, (total_val) in enumerate(total_public):
    if total_val > 0:
        ax2.text(i, total_val + 0.3, f'${total_val:.1f}B', 
                ha='center', va='bottom', fontsize=9, fontweight='bold')

# Formatting Panel B
ax2.set_ylabel('Public Burden (Billion USD)', fontsize=11, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(scenario_order, rotation=45, ha='right', fontsize=9)
ax2.set_yticks(np.arange(0, 80, 10))
ax2.tick_params(axis='both', which='major', length=4, width=1)
ax2.legend(loc='upper left', frameon=True, fontsize=8)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['left'].set_color('k')
ax2.spines['bottom'].set_color('k')
ax2.tick_params(axis='y', which='both', left=True, right=False)
ax2.grid(False)
ax2.text(-0.1, 1.05, 'b', transform=ax2.transAxes, fontsize=12, fontweight='bold', va='top')

# plt.tight_layout()
# plt.savefig('../results/figures/fig2_loss_institutional_stress.png', dpi=300, bbox_inches='tight')
# plt.show()

# print("\n✓ Figure saved to: ../results/figures/fig2_loss_institutional_stress.png")

In [ ]:
# Create 2-panel figure: Loss decomposition (a) and Institutional stress (b)
fig = plt.figure(figsize=(10, 3))
gs = fig.add_gridspec(1, 2, wspace=0.2)
ax1 = fig.add_subplot(gs[0, 0])  # Panel a: Loss decomposition
ax2 = fig.add_subplot(gs[0, 1])  # Panel b: Institutional stress

scenario_order = ['Great Miami', 'Andrew', 'Lake Okeechobee', 'Irma', 
                  'Great Miami then Andrew', 'Double Great Miami', 'Double Irma']

# ===== PANEL A: LOSS DECOMPOSITION =====
insured_wind_vals = [scenario_data[s]['insured_wind'] for s in scenario_order]
citizens_vals = [scenario_data[s]['citizens'] for s in scenario_order]
insured_flood_vals = [scenario_data[s]['insured_flood'] for s in scenario_order]
wind_under_vals = [scenario_data[s]['wind_under'] for s in scenario_order]
flood_under_vals = [scenario_data[s]['flood_under'] for s in scenario_order]

x_pos = np.arange(len(scenario_order))
width = 0.3

# Create stacked bars with intuitive colors: red for wind, blue for flood, orange for Citizens, grays for underinsured
p1 = ax1.bar(x_pos, insured_wind_vals, width, label='Insured Wind (Private)', color='#E74C3C')
p2 = ax1.bar(x_pos, citizens_vals, width, bottom=insured_wind_vals, 
            label='Citizens Wind', color='#F39C12')
bottom2 = np.array(insured_wind_vals) + np.array(citizens_vals)
p3 = ax1.bar(x_pos, insured_flood_vals, width, bottom=bottom2,
            label='Insured Flood (NFIP)', color='#3498DB')
bottom3 = bottom2 + np.array(insured_flood_vals)
p4 = ax1.bar(x_pos, wind_under_vals, width, bottom=bottom3,
            label='Un/Underinsured Wind', color='#95A5A6')
bottom4 = bottom3 + np.array(wind_under_vals)
p5 = ax1.bar(x_pos, flood_under_vals, width, bottom=bottom4,
            label='Un/Underinsured Flood', color='#BDC3C7')

# Calculate total losses and percentages
totals = (np.array(insured_wind_vals) + np.array(citizens_vals) + 
          np.array(insured_flood_vals) + np.array(wind_under_vals) + 
          np.array(flood_under_vals))

# Calculate two-way split: insured vs uninsured
insured_total = np.array(insured_wind_vals) + np.array(citizens_vals) + np.array(insured_flood_vals)
uninsured_total = np.array(wind_under_vals) + np.array(flood_under_vals)

insured_pct = (insured_total / totals) * 100
uninsured_pct = (uninsured_total / totals) * 100

# Add horizontal line and percentage labels
for i in range(len(scenario_order)):
    # Line separating insured from uninsured
    y_line = bottom3[i]  # Top of insured section
    ax1.plot([i - width/2, i + width/2], [y_line, y_line], 
            color='black', linewidth=1.5, linestyle='-', alpha=0.7)
    
    # Percentage labels
    # Insured percentage (middle of insured section)
    y_insured_mid = insured_total[i] / 2
    ax1.text(i + width/2 + 0.05, y_insured_mid, f'{insured_pct[i]:.0f}%', 
            ha='left', va='center', fontsize=7, fontweight='bold', color='#2C3E50')
    
    # Uninsured percentage (middle of uninsured section)
    y_uninsured_mid = bottom3[i] + uninsured_total[i] / 2
    ax1.text(i + width/2 + 0.05, y_uninsured_mid, f'{uninsured_pct[i]:.0f}%', 
            ha='left', va='center', fontsize=7, fontweight='bold', color='#7F8C8D')

# Add total labels on top of each bar
for i, total in enumerate(totals):
    ax1.text(i, total + 5, f'${total:.0f}B', ha='center', va='bottom', 
            fontweight='bold', fontsize=9)

# Formatting Panel A
ax1.set_ylabel('Loss (Billion USD)', fontsize=11, fontweight='bold')
#ax1.set_title('Loss Decomposition', fontsize=11, fontweight='bold', pad=10)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(scenario_order, rotation=45, ha='right', fontsize=9)
ax1.set_yticks(np.arange(0, 350, 50))
ax1.tick_params(axis='both', which='major', length=4, width=1)
ax1.legend(loc='upper left', frameon=True, fontsize=8, ncol=1, reverse=True)
ax1.grid(False)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['left'].set_color('k')
ax1.spines['bottom'].set_color('k')
ax1.tick_params(axis='y', which='both', left=True, right=False)
ax1.text(-0.1, 1.05, 'a', transform=ax1.transAxes, fontsize=12, fontweight='bold', va='top')

# ===== PANEL B: INSTITUTIONAL STRESS =====
# Extract institutional stress data for all scenarios
institutional_data = {}
for scenario in scenario_order:
    if scenario in df_baseline.columns:
        metrics = dict(zip(df_baseline['Metric'], df_baseline[scenario]))
        
        # Extract institutional stress metrics (in billions)
        fhcf_shortfall = metrics.get('FHCF shortfall (USD)', 0) / 1e9
        figa_residual = metrics.get('FIGA residual (USD)', 0) / 1e9
        citizens_deficit = metrics.get('Citizens residual (USD)', 0) / 1e9
        nfip_borrowed = metrics.get('NFIP Treasury borrowing (USD)', 0) / 1e9
        
        institutional_data[scenario] = {
            'fhcf_shortfall': fhcf_shortfall,
            'figa_residual': figa_residual,
            'citizens_deficit': citizens_deficit,
            'nfip_borrowed': nfip_borrowed
        }
fhcf_vals = np.array([institutional_data[s]['fhcf_shortfall'] for s in scenario_order])
figa_vals = np.array([institutional_data[s]['figa_residual'] for s in scenario_order])
citizens_vals_inst = np.array([institutional_data[s]['citizens_deficit'] for s in scenario_order])
nfip_vals = np.array([institutional_data[s]['nfip_borrowed'] for s in scenario_order])

# Calculate total public burden
total_public = fhcf_vals + figa_vals + citizens_vals_inst + nfip_vals

# Define colors for institutional stress
color_fhcf = "#DCB7EB"      # Purple for FHCF
color_figa = '#8E44AD'      # Darker purple for FIGA
color_citizens = '#F39C12'  # Orange for Citizens (matching previous)
color_nfip = '#3498DB'      # Blue for NFIP (matching flood)

# Create stacked bars)
bars1 = ax2.bar(x_pos, fhcf_vals, width, label='FHCF Shortfall', color=color_fhcf)
bars2 = ax2.bar(x_pos, figa_vals, width, bottom=fhcf_vals, label='FIGA Residual', color=color_figa)
bars3 = ax2.bar(x_pos, citizens_vals_inst, width, bottom=fhcf_vals + figa_vals, label='Citizens Deficit', color=color_citizens)
bars4 = ax2.bar(x_pos, nfip_vals, width, bottom=fhcf_vals + figa_vals + citizens_vals_inst, label='NFIP Borrowing', color=color_nfip)

# Add total public burden labels on top
for i, (total_val) in enumerate(total_public):
    if total_val > 0:
        ax2.text(i, total_val + 0.3, f'${total_val:.1f}B', 
                ha='center', va='bottom', fontsize=9, fontweight='bold')

# Formatting Panel B
ax2.set_ylabel('Public Burden (Billion USD)', fontsize=11, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(scenario_order, rotation=45, ha='right', fontsize=9)
ax2.set_yticks(np.arange(0, 80, 10))
ax2.tick_params(axis='both', which='major', length=4, width=1)
ax2.legend(loc='upper left', frameon=True, fontsize=8, reverse=True)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['left'].set_color('k')
ax2.spines['bottom'].set_color('k')
ax2.tick_params(axis='y', which='both', left=True, right=False)
ax2.grid(False)
ax2.text(-0.1, 1.05, 'b', transform=ax2.transAxes, fontsize=12, fontweight='bold', va='top')

plt.tight_layout()
plt.savefig('../results/figures/fig2_loss_institutional_stress.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to: ../results/figures/fig2_loss_institutional_stress.png")

## Comprehensive Metrics Table

Full summary table with all key metrics organized by category: loss decomposition, institutional stress indicators, capital adequacy, and stress ratios.

In [ ]:
# Create 2-panel figure: Private defaults (a) and Stress ratios (b)
fig = plt.figure(figsize=(10, 3))
gs = fig.add_gridspec(1, 2, wspace=0.2)
ax1 = fig.add_subplot(gs[0, 0])  # Panel a: Private defaults
ax2 = fig.add_subplot(gs[0, 1])  # Panel b: Stress ratios

# ===== PANEL A: PRIVATE DEFAULTS POST-GROUP =====
# Extract private defaults data
private_defaults = []
for scenario in scenario_order:
    if scenario in df_baseline.columns:
        metrics = dict(zip(df_baseline['Metric'], df_baseline[scenario]))
        defaults = metrics.get('Private defaults post-group (#)', 0)
        private_defaults.append(defaults)
    else:
        private_defaults.append(0)

private_defaults = np.array(private_defaults)

x_pos = np.arange(len(scenario_order))
width = 0.3

# Create simple bar plot
color_defaults = '#E74C3C'  # Red for defaults
bars = ax1.bar(x_pos, private_defaults, width, color=color_defaults, alpha=0.8)

# Add value labels on top of bars
for i, val in enumerate(private_defaults):
    if val > 0:
        ax1.text(i, val + 0.5, f'{int(val)}', 
                ha='center', va='bottom', fontsize=9, fontweight='bold')

# Formatting Panel A
ax1.set_ylabel('Number of Private Defaults', fontsize=11, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(scenario_order, rotation=45, ha='right', fontsize=9)
ax1.set_yticks(np.arange(0, max(private_defaults) + 5, 5))
ax1.tick_params(axis='both', which='major', length=4, width=1)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['left'].set_color('k')
ax1.spines['bottom'].set_color('k')
ax1.tick_params(axis='y', which='both', left=True, right=False)
ax1.grid(False)
ax1.text(-0.1, 1.05, 'a', transform=ax1.transAxes, fontsize=12, fontweight='bold', va='top')

# ===== PANEL B: STRESS RATIOS (GROUPED) =====
# Extract stress ratio data for all scenarios
stress_ratio_data = {}
for scenario in scenario_order:
    if scenario in df_baseline.columns:
        metrics = dict(zip(df_baseline['Metric'], df_baseline[scenario]))
        
        # Extract stress ratios (as factors: e.g., 10.5 = 10.5 years of premiums for NFIP)
        fhcf_ratio = metrics.get('FHCF utilization ratio (%)', 0)
        citizens_ratio = metrics.get('Citizens assessment stress ratio (%)', 0)
        figa_ratio = metrics.get('FIGA stress ratio (%)', 0)
        nfip_ratio = metrics.get('NFIP Florida stress ratio (%)', 0)
        
        stress_ratio_data[scenario] = {
            'fhcf': fhcf_ratio,
            'citizens': citizens_ratio,
            'figa': figa_ratio,
            'nfip': nfip_ratio
        }

fhcf_ratios = np.array([stress_ratio_data[s]['fhcf'] for s in scenario_order])
citizens_ratios = np.array([stress_ratio_data[s]['citizens'] for s in scenario_order])
figa_ratios = np.array([stress_ratio_data[s]['figa'] for s in scenario_order])
nfip_ratios = np.array([stress_ratio_data[s]['nfip'] for s in scenario_order])

width = 0.2  # Narrower bars for grouped display

# Define colors matching previous figures
color_fhcf = "#DCB7EB"      # Purple for FHCF
color_citizens = '#F39C12'  # Orange for Citizens
color_figa = '#8E44AD'      # Darker purple for FIGA
color_nfip = '#3498DB'      # Blue for NFIP

# Create grouped bars side-by-side
bars1 = ax2.bar(x_pos - 1.5*width, fhcf_ratios, width, label='FHCF Utilization', color=color_fhcf)
bars2 = ax2.bar(x_pos - 0.5*width, citizens_ratios, width, 
                label='Citizens Assessment Stress', color=color_citizens)
bars3 = ax2.bar(x_pos + 0.5*width, figa_ratios, width, 
                label='FIGA Stress', color=color_figa)
bars4 = ax2.bar(x_pos + 1.5*width, nfip_ratios, width, 
                label='NFIP Florida Stress', color=color_nfip)

# # Add value labels on top of each bar (only if value > 0)
# for i in range(len(scenario_order)):
#     if fhcf_ratios[i] > 0:
#         ax2.text(i - 1.5*width, fhcf_ratios[i] + 0.5, f'{fhcf_ratios[i]:.0f}', 
#                 ha='center', va='bottom', fontsize=7)
#     if citizens_ratios[i] > 0:
#         ax2.text(i - 0.5*width, citizens_ratios[i] + 0.5, f'{citizens_ratios[i]:.0f}', 
#                 ha='center', va='bottom', fontsize=7)
#     if figa_ratios[i] > 0:
#         ax2.text(i + 0.5*width, figa_ratios[i] + 0.5, f'{figa_ratios[i]:.0f}', 
#                 ha='center', va='bottom', fontsize=7)
#     if nfip_ratios[i] > 0:
#         ax2.text(i + 1.5*width, nfip_ratios[i] + 0.5, f'{nfip_ratios[i]:.0f}', 
#                 ha='center', va='bottom', fontsize=7)

# Formatting Panel B
ax2.set_ylabel('Stress Factor', fontsize=11, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(scenario_order, rotation=45, ha='right', fontsize=9)
ax2.set_ylim(0, max(max(fhcf_ratios), max(citizens_ratios), max(figa_ratios), max(nfip_ratios)) * 1.15)
ax2.tick_params(axis='both', which='major', length=4, width=1)
ax2.legend(loc='upper left', frameon=True, fontsize=8, ncol=1)
ax2.grid(False)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['left'].set_color('k')
ax2.spines['bottom'].set_color('k')
ax2.tick_params(axis='y', which='both', left=True, right=False)
ax2.text(-0.1, 1.05, 'b', transform=ax2.transAxes, fontsize=12, fontweight='bold', va='top')

plt.tight_layout()
plt.savefig('../results/figures/fig3_stress_ratios_defaults.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to: ../results/figures/fig3_stress_ratios_defaults.png")

In [ ]:
# Load uncertainty data (5th and 95th percentiles)
df_p5 = pd.read_excel(BASELINE_FILE, sheet_name='5th Percentile')
df_p95 = pd.read_excel(BASELINE_FILE, sheet_name='95th Percentile')

# Build comprehensive table with all metrics organized by category
comprehensive_metrics = []

# ===== LOSS DECOMPOSITION SECTION =====
comprehensive_metrics.append({
    'Category': 'LOSS DECOMPOSITION',
    'Metric': '',
    **{scenario: '' for scenario in scenario_order}
})

# Add loss decomposition metrics from the figure
loss_decomp_metrics = [
    ('Total Loss (USD)', 'Total loss (USD)'),
    ('Insured Wind - Private (USD)', None),  # Calculated
    ('Citizens Wind (USD)', 'Citizens loss gross (USD)'),
    ('Insured Flood - NFIP (USD)', None),  # Calculated
    ('Un/Underinsured Wind (USD)', 'Un/underinsured wind (USD)'),
    ('Un/Underinsured Flood (USD)', 'Un/underinsured flood (USD)'),
]

for display_name, metric_key in loss_decomp_metrics:
    row_data = {'Category': '', 'Metric': display_name}
    
    for scenario in scenario_order:
        if scenario in df_baseline.columns:
            metrics_mean = dict(zip(df_baseline['Metric'], df_baseline[scenario]))
            metrics_p5 = dict(zip(df_p5['Metric'], df_p5[scenario]))
            metrics_p95 = dict(zip(df_p95['Metric'], df_p95[scenario]))
            
            if metric_key is None:
                # Calculate derived metrics
                if 'Insured Wind - Private' in display_name:
                    wind_gross = metrics_mean.get('Wind loss gross (USD)', 0)
                    citizens = metrics_mean.get('Citizens loss gross (USD)', 0)
                    wind_under = metrics_mean.get('Un/underinsured wind (USD)', 0)
                    mean_val = wind_gross - citizens - wind_under
                    
                    wind_gross_p5 = metrics_p5.get('Wind loss gross (USD)', 0)
                    citizens_p5 = metrics_p5.get('Citizens loss gross (USD)', 0)
                    wind_under_p5 = metrics_p5.get('Un/underinsured wind (USD)', 0)
                    p5_val = wind_gross_p5 - citizens_p5 - wind_under_p5
                    
                    wind_gross_p95 = metrics_p95.get('Wind loss gross (USD)', 0)
                    citizens_p95 = metrics_p95.get('Citizens loss gross (USD)', 0)
                    wind_under_p95 = metrics_p95.get('Un/underinsured wind (USD)', 0)
                    p95_val = wind_gross_p95 - citizens_p95 - wind_under_p95
                    
                elif 'Insured Flood' in display_name:
                    flood_gross = metrics_mean.get('Flood loss gross (USD)', 0)
                    flood_under = metrics_mean.get('Un/underinsured flood (USD)', 0)
                    mean_val = flood_gross - flood_under
                    
                    flood_gross_p5 = metrics_p5.get('Flood loss gross (USD)', 0)
                    flood_under_p5 = metrics_p5.get('Un/underinsured flood (USD)', 0)
                    p5_val = flood_gross_p5 - flood_under_p5
                    
                    flood_gross_p95 = metrics_p95.get('Flood loss gross (USD)', 0)
                    flood_under_p95 = metrics_p95.get('Un/underinsured flood (USD)', 0)
                    p95_val = flood_gross_p95 - flood_under_p95
            else:
                mean_val = metrics_mean.get(metric_key, 0)
                p5_val = metrics_p5.get(metric_key, 0)
                p95_val = metrics_p95.get(metric_key, 0)
            
            # Convert to billions
            mean_val /= 1e9
            p5_val /= 1e9
            p95_val /= 1e9
            row_data[scenario] = f"${mean_val:.1f}B (${p5_val:.1f}-${p95_val:.1f}B)"
        else:
            row_data[scenario] = 'N/A'
    
    comprehensive_metrics.append(row_data)

# ===== INSTITUTIONAL STRESS SECTION =====
comprehensive_metrics.append({
    'Category': 'INSTITUTIONAL STRESS',
    'Metric': '',
    **{scenario: '' for scenario in scenario_order}
})

public_burden_metrics = [
    ('Total Public Burden (USD)', None),  # Calculated
    ('FHCF Shortfall (USD)', 'FHCF shortfall (USD)'),
    ('FIGA Residual (USD)', 'FIGA residual (USD)'),
    ('Citizens Deficit (USD)', 'Citizens residual (USD)'),
    ('NFIP Treasury Borrowing (USD)', 'NFIP Treasury borrowing (USD)'),
]

for display_name, metric_key in public_burden_metrics:
    row_data = {'Category': '', 'Metric': display_name}
    
    for scenario in scenario_order:
        if scenario in df_baseline.columns:
            metrics_mean = dict(zip(df_baseline['Metric'], df_baseline[scenario]))
            metrics_p5 = dict(zip(df_p5['Metric'], df_p5[scenario]))
            metrics_p95 = dict(zip(df_p95['Metric'], df_p95[scenario]))
            
            if metric_key is None:
                # Calculate total public burden
                fhcf = metrics_mean.get('FHCF shortfall (USD)', 0)
                figa = metrics_mean.get('FIGA residual (USD)', 0)
                citizens = metrics_mean.get('Citizens residual (USD)', 0)
                nfip = metrics_mean.get('NFIP Treasury borrowing (USD)', 0)
                mean_val = fhcf + figa + citizens + nfip
                
                fhcf_p5 = metrics_p5.get('FHCF shortfall (USD)', 0)
                figa_p5 = metrics_p5.get('FIGA residual (USD)', 0)
                citizens_p5 = metrics_p5.get('Citizens residual (USD)', 0)
                nfip_p5 = metrics_p5.get('NFIP Treasury borrowing (USD)', 0)
                p5_val = fhcf_p5 + figa_p5 + citizens_p5 + nfip_p5
                
                fhcf_p95 = metrics_p95.get('FHCF shortfall (USD)', 0)
                figa_p95 = metrics_p95.get('FIGA residual (USD)', 0)
                citizens_p95 = metrics_p95.get('Citizens residual (USD)', 0)
                nfip_p95 = metrics_p95.get('NFIP Treasury borrowing (USD)', 0)
                p95_val = fhcf_p95 + figa_p95 + citizens_p95 + nfip_p95
            else:
                mean_val = metrics_mean.get(metric_key, 0)
                p5_val = metrics_p5.get(metric_key, 0)
                p95_val = metrics_p95.get(metric_key, 0)
            
            # Convert to billions
            mean_val /= 1e9
            p5_val /= 1e9
            p95_val /= 1e9
            row_data[scenario] = f"${mean_val:.1f}B (${p5_val:.1f}-${p95_val:.1f}B)"
        else:
            row_data[scenario] = 'N/A'
    
    comprehensive_metrics.append(row_data)

# ===== CAPITAL AND DEFAULT ASSESSMENT SECTION =====
comprehensive_metrics.append({
    'Category': 'CAPITAL AND DEFAULT ASSESSMENT',
    'Metric': '',
    **{scenario: '' for scenario in scenario_order}
})

capital_metrics = [
    ('Private Defaults Post-Group (#)', 'Private defaults post-group (#)'),
    ('Largest Single-Entity Deficit (USD)', 'Largest single-entity deficit (USD)'),
]

for display_name, metric_key in capital_metrics:
    row_data = {'Category': '', 'Metric': display_name}
    
    for scenario in scenario_order:
        if scenario in df_baseline.columns:
            metrics_mean = dict(zip(df_baseline['Metric'], df_baseline[scenario]))
            metrics_p5 = dict(zip(df_p5['Metric'], df_p5[scenario]))
            metrics_p95 = dict(zip(df_p95['Metric'], df_p95[scenario]))
            
            mean_val = metrics_mean.get(metric_key, 0)
            p5_val = metrics_p5.get(metric_key, 0)
            p95_val = metrics_p95.get(metric_key, 0)
            
            if 'USD' in display_name:
                # Convert to billions
                mean_val /= 1e9
                p5_val /= 1e9
                p95_val /= 1e9
                row_data[scenario] = f"${mean_val:.1f}B (${p5_val:.1f}-${p95_val:.1f}B)"
            else:
                # Integer count
                row_data[scenario] = f"{int(mean_val)} ({int(p5_val)}-{int(p95_val)})"
        else:
            row_data[scenario] = 'N/A'
    
    comprehensive_metrics.append(row_data)

# ===== STRESS RATIOS SECTION =====
comprehensive_metrics.append({
    'Category': 'STRESS RATIOS',
    'Metric': '',
    **{scenario: '' for scenario in scenario_order}
})

stress_ratio_metrics = [
    ('FHCF Utilization Factor', 'FHCF utilization ratio (%)'),
    ('Citizens Assessment Stress Factor', 'Citizens assessment stress ratio (%)'),
    ('FIGA Stress Factor', 'FIGA stress ratio (%)'),
    ('NFIP Florida Stress Factor', 'NFIP Florida stress ratio (%)'),
]

for display_name, metric_key in stress_ratio_metrics:
    row_data = {'Category': '', 'Metric': display_name}
    
    for scenario in scenario_order:
        if scenario in df_baseline.columns:
            metrics_mean = dict(zip(df_baseline['Metric'], df_baseline[scenario]))
            metrics_p5 = dict(zip(df_p5['Metric'], df_p5[scenario]))
            metrics_p95 = dict(zip(df_p95['Metric'], df_p95[scenario]))
            
            # Display as stress factors (e.g., 10.5 = 10.5 years for NFIP)
            mean_val = metrics_mean.get(metric_key, 0)
            p5_val = metrics_p5.get(metric_key, 0)
            p95_val = metrics_p95.get(metric_key, 0)
            
            row_data[scenario] = f"{mean_val:.2f} ({p5_val:.2f}-{p95_val:.2f})"
        else:
            row_data[scenario] = 'N/A'
    
    comprehensive_metrics.append(row_data)

# Create DataFrame
df_comprehensive = pd.DataFrame(comprehensive_metrics)

# Display the table
print("\n" + "="*160)
print("COMPREHENSIVE METRICS TABLE - ALL SCENARIOS")
print("Values shown as: Mean (5th-95th Percentile)")
print("="*160)
print(df_comprehensive.to_string(index=False))
print("="*160)

# Save to CSV with proper encoding
df_comprehensive.to_csv('../results/tables/comprehensive_metrics_all_scenarios.csv', index=False, encoding='utf-8')
print("\n✓ Comprehensive table saved to: ../results/tables/comprehensive_metrics_all_scenarios.csv")